In [4]:
# 1. Install library Gradio dan openpyxl
!pip install -q gradio openpyxl

import gradio as gr
import pandas as pd
from collections import Counter
import re

# 2. Fungsi untuk menampilkan seluruh frekuensi kata
def hitung_seluruh_kata(file_obj):
    if file_obj is None:
        return "⚠️ Halo! Silakan unggah file Excel atau teks terlebih dahulu ya."

    try:
        nama_file = file_obj.name
        semua_teks = ""

        # Cek apakah file berupa Excel
        if nama_file.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(nama_file)
            pilihan_kolom = ['text', 'comment', 'comments', 'komentar', 'isi', 'content', 'tweet', 'pesan']
            kolom_terpilih = None

            for col in pilihan_kolom:
                if col in df.columns:
                    kolom_terpilih = col
                    break

            if kolom_terpilih is None:
                for col in df.columns:
                    if df[col].dtype == 'object':
                        kolom_terpilih = col
                        break

            if kolom_terpilih is not None:
                semua_teks = " ".join(df[kolom_terpilih].dropna().astype(str))
            else:
                return "❌ Maaf, program tidak menemukan kolom teks di file Excel ini."
        else:
            with open(nama_file, 'r', encoding='utf-8', errors='ignore') as f:
                semua_teks = f.read()

        # Proses pembersihan teks dan hitung kata
        kata_kata = re.findall(r'\b\w+\b', semua_teks.lower())
        if not kata_kata:
            return "⚠️ File terbaca, tapi tidak ada kata yang bisa dihitung."

        frekuensi = Counter(kata_kata)

        # Mengambil SELURUH kata, diurutkan dari yang terbanyak
        # .most_common() tanpa angka di dalam kurung akan mengambil semua data
        seluruh_hasil = frekuensi.most_common()

        # Format hasil ke dalam teks rapi
        output_teks = f"🎉 Berhasil! Total ada {len(seluruh_hasil)} kata unik yang ditemukan:\n" + "="*45 + "\n"
        for idx, (kata, jumlah) in enumerate(seluruh_hasil, 1):
            output_teks += f"{idx}. {kata} : {jumlah}x\n"

        return output_teks

    except Exception as e:
        return f"❌ Terjadi kendala saat membaca file.\n(Detail: {str(e)})"

# 3. Membangun Antarmuka Sederhana (UI Wrapper)
with gr.Blocks() as demo:
    gr.Markdown("# 📊 Penghitung Seluruh Frekuensi Kata (Excel)")
    gr.Markdown("Aplikasi ini akan menghitung dan menampilkan **semua** kata unik dari file Excel Anda secara berurutan.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="Unggah File Excel (.xlsx / .xls) atau TXT")
            submit_btn = gr.Button("🚀 Proses Semua Kata", variant="primary")

        with gr.Column():
            # Kotak teks diperbesar ukuran barisnya agar menampung banyak data
            output_box = gr.Textbox(label="Hasil Analisis Seluruh Kata", lines=20)

    submit_btn.click(fn=hitung_seluruh_kata, inputs=file_input, outputs=output_box)

# 4. Jalankan aplikasi
demo.launch(inline=False, share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://083708ebe9e120d93c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
